In [1]:
%pip install -U "jax[cuda12]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.3/83.3 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 51.6 MB/s eta 0:00:00
  Attempting uninstall: jax-cuda12-pjrt
    Found existing installation: jax-cuda12-pjrt 0.7.2
    Uninstalling jax-cuda12-pjrt-0.7.2:
      Successfully uninstalled jax-cuda12-pjrt-0.7.2
  Attempting uninstall: nvidia-cuda-nvcc-cu12
    Found existing installation: nvidia-cuda-nvcc-cu12 12.5.82
    Uninstalling nvidia-cuda-nvcc-cu12-12.5.82:
      Successfully uninstalled nvidia-cuda-nvcc-cu12-12.5.82
  Attempting uninstall: jax-cuda12-plugin
    Found existing installation: jax-cuda12-plugin 0.7.2
    Uninstalling jax-cuda12-plugin-0.7.2:
      Successfully uninstalled jax-cuda12-plugin-0.7.2
  Att

In [2]:
import os

In [3]:
os.environ['JAX_PLATFORM_NAME'] = 'gpu' 
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false' # to prevent jax from preallocsting 75% vram.

In [4]:
import jax
import jax.numpy as jnp 

In [5]:
import jax.lax as lax
import tiktoken
import pickle
import math
from jax import vmap

In [6]:
jax.devices()

[CudaDevice(id=0), CudaDevice(id=1)]

In [7]:
enc = tiktoken.get_encoding("gpt2")

In [8]:
with open('/kaggle/input/datasets/pandeyps/shakespeare/shakespeare.txt', 'r') as f:
    text = f.read()
tokens = enc.encode(text)
data = jnp.array(tokens)

In [9]:
data

Array([  220,  3574, 37063, ..., 10970, 23578,   198], dtype=int32)

In [10]:
class args:
    vocab_size = enc.n_vocab
    dim = 256
    n_layers = 2
    n_heads = 4
    n_kv_heads = 2
    max_seq_len = 128
    batch_size = 16
    learning_rate = 3e-4
    dropout_rate = 0.0

In [11]:
config = args()

In [12]:
from jax import random

In [13]:
def save_params(params, filepath):
    numpy_params = jax.tree.map(lambda x: x.copy(), params)
    with open(filepath, 'wb') as f:
        pickle.dump(numpy_params, f)

def load_params(filepath):
    with open(filepath, 'rb') as f:
        numpy_params = pickle.load(f)
        # jax arrays
    params = jax.tree.map(lambda x: jnp.array(x), numpy_params)
    return params

In [14]:
def get_batch(key, data, batch_size, seq_len):
    ix = random.randint(key, (batch_size,), 0, len(data) - seq_len)

    x = vmap(lambda i: lax.dynamic_slice(data, (i,), (seq_len,)))(ix)
    y = vmap(lambda i: lax.dynamic_slice(data, (i + 1,), (seq_len,)))(ix)

    return x, y

In [15]:
def generate(params, prompt_tokens, max_new_tokens, config, key):
    x = jnp.array(prompt_tokens)
    for _ in range(max_new_tokens):
        x_crop = x[-config.max_seq_len:]
        logits, _ = model_forward(params, x_crop[None, :], config)
        logits = logits[0, -1, :] 
        
        # advance the key so each token is sampled differently
        key, subkey = random.split(key) 
        next_token = random.categorical(subkey, logits, shape=(1,))[0]
    
        x = jnp.concatenate([x, jnp.array([next_token])])
    return x.tolist()

In [16]:
def rms_norm(x, weight, eps=1e-6):
    var = jnp.mean(jnp.square(x), axis=-1, keepdims = True)
    return x*weight*jnp.reciprocal(jnp.sqrt(var+eps))

In [17]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (jnp.arange(0, dim // 2, dtype=jnp.float32) / dim))
    t = jnp.arange(end, dtype=jnp.float32)
    freqs = jnp.outer(t, freqs)
    return jnp.complex64(jnp.exp(1j * freqs))

In [18]:
def apply_rotary_emb(xq, xk, freqs_cis):
    xq_r, xk_r = jnp.reshape(xq, (*xq.shape[:-1], -1, 2)), jnp.reshape(xk, (*xk.shape[:-1], -1, 2))
    xq_complex = jnp.complex64(xq_r[..., 0] + 1j * xq_r[..., 1])
    xk_complex = jnp.complex64(xk_r[..., 0] + 1j * xk_r[..., 1])
    freqs_cis = jnp.reshape(freqs_cis, (1, freqs_cis.shape[0], 1, freqs_cis.shape[1]))
    xq_out = xq_complex * freqs_cis
    xk_out = xk_complex * freqs_cis
    xq = jnp.stack([jnp.real(xq_out), jnp.imag(xq_out)], axis=-1).reshape(xq.shape)
    xk = jnp.stack([jnp.real(xk_out), jnp.imag(xk_out)], axis=-1).reshape(xk.shape)
    return xq, xk

In [19]:
def repeat_kv(x, n_rep):
    return x if n_rep == 1 else jnp.repeat(x, n_rep, axis=2)

In [20]:
def init_weight(key, shape, scale=None):
    scale = 1.0 / math.sqrt(shape[0]) if scale is None else scale
    return jax.random.normal(key, shape) * scale

In [21]:
# attention has 4 params:trainable
def init_attention_weights(key, dim, n_heads, n_kv_heads):
    keys = jax.random.split(key, 4)
    head_dim = dim // n_heads
    return {
        'wq': init_weight(keys[0], (dim, n_heads * head_dim)),
        'wk': init_weight(keys[1], (dim, n_kv_heads * head_dim)),
        'wv': init_weight(keys[2], (dim, n_kv_heads * head_dim)),
        'wo': init_weight(keys[3], (n_heads * head_dim, dim))
    }

In [22]:
def init_ffn_weights(key, dim):
    keys = jax.random.split(key, 3)
    return {
        'w1': init_weight(keys[0], (dim, 4 * dim)),
        'w2': init_weight(keys[1], (4 * dim, dim)),
        'w3': init_weight(keys[2], (dim, 4 * dim))}

In [23]:
def init_transformer_block(key, dim, n_heads, n_kv_heads):
    keys = jax.random.split(key, 4)
    return {
        'attention': init_attention_weights(keys[0], dim, n_heads, n_kv_heads),
        'ffn': init_ffn_weights(keys[1], dim),
        'attention_norm': init_weight(keys[2], (dim,), scale=1.0),
        'ffn_norm': init_weight(keys[3], (dim,), scale=1.0)}

In [24]:
def init_model_params(key, vocab_size, dim, n_layers, n_heads, n_kv_heads):
    keys = jax.random.split(key, 4)
    params = {
        'token_embedding': init_weight(keys[0], (vocab_size, dim)),
        'norm_f': init_weight(keys[1], (dim,), scale=1.0),
        'output': init_weight(keys[2], (dim, vocab_size))
    }
    block_keys = jax.random.split(keys[3], n_layers)
    params['blocks'] = [init_transformer_block(k, dim, n_heads, n_kv_heads) for k in block_keys]
    return params

In [25]:
#GQA
def attention(params, x, mask, freqs_cis, n_heads, n_kv_heads, cache=None, position=0):
    B, T, C = x.shape
    head_dim = C // n_heads
    q = jnp.dot(x, params['wq']).reshape(B, T, n_heads, head_dim)
    k = jnp.dot(x, params['wk']).reshape(B, T, n_kv_heads, head_dim)
    v = jnp.dot(x, params['wv']).reshape(B, T, n_kv_heads, head_dim)
    q, k = apply_rotary_emb(q, k, freqs_cis[position:position + T])
    if cache is not None:
        k = jnp.concatenate([cache[0], k], axis=1)
        v = jnp.concatenate([cache[1], v], axis=1)
    new_cache = (k, v)
    k = repeat_kv(k, n_heads // n_kv_heads)
    v = repeat_kv(v, n_heads // n_kv_heads)
    q, k, v = map(lambda x: x.transpose(0, 2, 1, 3), (q, k, v))
    scores = jnp.matmul(q, k.transpose(0, 1, 3, 2)) / math.sqrt(head_dim)
    if mask is not None:
        scores = scores + mask[:, :, :T, :T]
    scores = jax.nn.softmax(scores, axis=-1)
    output = jnp.matmul(scores, v)
    output = output.transpose(0, 2, 1, 3).reshape(B, T, -1)
    return jnp.dot(output, params['wo']), new_cache

In [26]:
def feed_forward(params, x):
    return jnp.dot(jax.nn.silu(jnp.dot(x, params['w3'])) * jnp.dot(x, params['w1']), params['w2'])

In [27]:
def transformer_block(params, x, mask, freqs_cis, n_heads, n_kv_heads, cache=None, position=0, training=False, dropout_rate=0.0, key=None):
    attn_output, new_cache = attention(params['attention'], rms_norm(x, params['attention_norm']), mask, freqs_cis, n_heads, n_kv_heads, cache, position)
    if training:
        dropout_key, key = jax.random.split(key)
        attn_output = jax.random.bernoulli(dropout_key, 1-dropout_rate, shape=attn_output.shape) * attn_output / (1-dropout_rate)
    h = x + attn_output
    ffn_output = feed_forward(params['ffn'], rms_norm(h, params['ffn_norm']))
    if training:
        dropout_key, key = jax.random.split(key)
        ffn_output = jax.random.bernoulli(dropout_key, 1-dropout_rate, shape=ffn_output.shape) * ffn_output / (1-dropout_rate)
    out = h + ffn_output
    return out, new_cache

In [28]:
def model_forward(params, inputs, config, cache=None, position=0):
    B, T = inputs.shape
    h = params['token_embedding'][inputs]
    freqs_cis = precompute_freqs_cis(config.dim // config.n_heads, config.max_seq_len)
    mask = jnp.tril(jnp.ones((config.max_seq_len, config.max_seq_len)))
    mask = jnp.where(mask == 0, -1e9, 0.0)
    mask = mask.astype(h.dtype)
    mask = mask[None, None, :, :]
    new_caches = []
    for i, block in enumerate(params['blocks']):
        layer_cache = cache[i] if cache is not None else None
        h, layer_cache = transformer_block(block, h, mask, freqs_cis, config.n_heads, config.n_kv_heads, layer_cache, position, training=False, dropout_rate=config.dropout_rate)
        new_caches.append(layer_cache)
    h = rms_norm(h, params['norm_f'])
    logits = jnp.dot(h, params['output'])
    return logits, new_caches

In [29]:
def compute_loss(params, batch):
    inputs, targets = batch
    logits, _ = model_forward(params, inputs, config)
    logits = logits.reshape(-1, config.vocab_size)
    targets = targets.reshape(-1)
    loss = -jnp.mean(
        jnp.take_along_axis(
            jax.nn.log_softmax(logits),
            targets[:, None],
            axis=1
        )
    )
    return loss

In [30]:
@jax.jit
def update_step(params, batch):
    loss, grads = jax.value_and_grad(compute_loss)(params, batch)
    params = jax.tree.map(
        lambda p, g: p - config.learning_rate * g,
        params,
        grads
    )
    return params, loss

In [31]:
key = random.PRNGKey(0)
params = init_model_params(
    key=key,
    vocab_size=config.vocab_size,
    dim=config.dim,
    n_layers=config.n_layers,
    n_heads=config.n_heads,
    n_kv_heads=config.n_kv_heads
)

In [32]:
def train(num_epochs=30, steps_per_epoch=1000):
    key = random.PRNGKey(0)
    params_state = params 

    epoch_losses = []

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch + 1}/{num_epochs}")
        print("-" * 50)

        epoch_loss = 0.0
        for step in range(steps_per_epoch):

            key, batch_key = random.split(key)

            batch = get_batch(batch_key, data, config.batch_size, config.max_seq_len)

            #update model
            params_state, loss = update_step(params_state, batch)
            epoch_loss += loss


            if step % 100 == 0:
                print(f"epoch {epoch + 1}, step {step}/{steps_per_epoch}: loss = {loss:.4f}")


        avg_epoch_loss = epoch_loss / steps_per_epoch
        epoch_losses.append(avg_epoch_loss)

        print(f"\nepoch {epoch + 1} | average loss: {avg_epoch_loss:.4f}")


        if (epoch + 1) % 5 == 0:
            save_params(params_state, f'model_checkpoint_epoch_{epoch+1}.pkl')


    print("Loss by epoch:")
    for epoch, loss in enumerate(epoch_losses, 1):
        print(f"Epoch {epoch}: {loss:.4f}")

# save
    save_params(params_state, 'model_final.pkl')
    return params_state

In [33]:
trained_params = train()


Epoch 1/30
--------------------------------------------------
epoch 1, step 0/1000: loss = 11.4643
epoch 1, step 100/1000: loss = 8.4700
epoch 1, step 200/1000: loss = 8.5789
epoch 1, step 300/1000: loss = 8.1109
epoch 1, step 400/1000: loss = 8.0799
epoch 1, step 500/1000: loss = 7.9408
epoch 1, step 600/1000: loss = 8.2755
epoch 1, step 700/1000: loss = 7.6022
epoch 1, step 800/1000: loss = 7.6641
epoch 1, step 900/1000: loss = 7.3740

epoch 1 | average loss: 8.0950

Epoch 2/30
--------------------------------------------------
epoch 2, step 0/1000: loss = 7.5967
epoch 2, step 100/1000: loss = 6.9690
epoch 2, step 200/1000: loss = 7.2289
epoch 2, step 300/1000: loss = 7.2645
epoch 2, step 400/1000: loss = 7.4226
epoch 2, step 500/1000: loss = 7.1935
epoch 2, step 600/1000: loss = 7.4607
epoch 2, step 700/1000: loss = 7.4150
epoch 2, step 800/1000: loss = 6.9158
epoch 2, step 900/1000: loss = 7.2207

epoch 2 | average loss: 7.4080

Epoch 3/30
-----------------------------------------

In [34]:
# inference
prompt = tokens[:10] if 'tokens' in locals() else [1, 2, 3]
key, gen_key = random.split(key)
output_tokens = generate(params, prompt, 20, config, gen_key)
print(f"tokens: {output_tokens}")
print(f"decoded: {enc.decode(output_tokens)}")

tokens: [220, 3574, 37063, 301, 8109, 356, 6227, 2620, 11, 198, 35816, 44724, 30597, 32235, 2495, 33906, 12234, 47409, 44627, 37194, 30610, 43283, 4146, 30120, 10617, 1199, 6604, 5503, 14382, 37591]
decoded:   From fairest creatures we desire increase,
 VuSTATE WellingtonREAM prettyVa hash whine++++++++++++++++ Cutler768 obnoxiousIL407 qualifiedWhclaim stressplane feudal
